In [1]:
import os
import sys
from dotenv import load_dotenv

# Add parent directory to path to import azure_openai_config
sys.path.append(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', '..'))

load_dotenv()

from azure_openai_config import setup_azure_openai, get_azure_openai_model
setup_azure_openai()


In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()

In [3]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    get_azure_openai_model(),
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions."
)

In [8]:
# Inspect the tools to see what parameters they expect
import json
for tool in tools:
    print(f"Tool: {tool.name}")
    print(f"Description: {tool.description}")
    if hasattr(tool, 'args_schema'):
        print(f"Parameters: {json.dumps(tool.args_schema, indent=2)}")
    print("-" * 50)

Tool: search-flight
Description: 
# Search for a flight

## Description

Uses the Kiwi API to search for available flights between two locations on a specific date.

## How it works

The tool will:
1. Search for matching locations to resolve airport codes
2. Find available flights for the specified route and date range

## Method

Call this tool whenever a user wants to search for flights, regardless of whether they provided exact airport codes or just city names.

You should display the returned results in a markdown table format: Group the results by price (those who are the cheapest), duration (those who are the shortest, i.e. have the smallest 'totalDurationInSeconds') and the rest (those that could still be interesting).

Always display for each flight in order:
  - In the 1st column: The departure and arrival airports, including layovers (e.g. "Paris CDG → Barcelona BCN → Lisbon LIS")
  - In the 2nd column: The departure and arrival dates & times in the local timezones, and durat

In [11]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

try:
    response = await agent.ainvoke(
        {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on March 31st")]},
        config
    )
except Exception as e:
    print(f"Error type: {type(e).__name__}")
    print(f"Error message: {str(e)}")
    import traceback
    traceback.print_exc()

Error type: BadRequestError
Error message: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_PB9QikYiIU9ZiLUekWpS84bG", 'type': 'invalid_request_error', 'param': 'messages.[3].role', 'code': None}}


Traceback (most recent call last):
  File "C:\Users\manikanta.reddy\AppData\Local\Temp\ipykernel_17872\4203388311.py", line 6, in <module>
    response = await agent.ainvoke(
               ^^^^^^^^^^^^^^^^^^^^
    ...<2 lines>...
    )
    ^
  File "c:\Users\manikanta.reddy\source\repos\lca-lc-foundations\.venv\Lib\site-packages\langgraph\pregel\main.py", line 3158, in ainvoke
    async for chunk in self.astream(
    ...<29 lines>...
            chunks.append(chunk)
  File "c:\Users\manikanta.reddy\source\repos\lca-lc-foundations\.venv\Lib\site-packages\langgraph\pregel\main.py", line 2971, in astream
    async for _ in runner.atick(
    ...<13 lines>...
            yield o
  File "c:\Users\manikanta.reddy\source\repos\lca-lc-foundations\.venv\Lib\site-packages\langgraph\pregel\_runner.py", line 304, in atick
    await arun_with_retry(
    ...<15 lines>...
    )
  File "c:\Users\manikanta.reddy\source\repos\lca-lc-foundations\.venv\Lib\site-packages\langgraph\pregel\_retry.py", line 1

In [ ]:
from pprint import pprint

pprint(response)

In [ ]:
print(response["messages"][-1].content)